# Geneformer Embeddings

**Before running:**
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Upload `prepared_GSE120575.h5ad` using the file panel on the left
3. Run all cells in order
4. Download `melanoma_tcells_embeddings.h5ad` from the file panel when done

In [ ]:
# Cell 1: Install dependencies
!pip install anndata --quiet
!git clone https://huggingface.co/ctheodoris/Geneformer
!pip install -e Geneformer/ --quiet

In [ ]:
# Cell 2: Confirm GPU and imports
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

import os
import shutil
import numpy as np
import pandas as pd
import anndata as ad
from geneformer import TranscriptomeTokenizer, EmbExtractor
print('All imports ok')

In [ ]:
# Cell 3: Paths
INPUT_FILE       = '/content/prepared_GSE120575.h5ad'
GENEFORMER_DIR   = '/content/Geneformer/Geneformer-V2-104M'
GENE_MEDIAN_FILE = '/content/Geneformer/geneformer/gene_median_dictionary_gc104M.pkl'
TOKEN_DICT_FILE  = '/content/Geneformer/geneformer/token_dictionary_gc104M.pkl'
OUTPUT_DIR       = '/content/output/'
OUTPUT_H5AD      = '/content/output/melanoma_tcells_embeddings.h5ad'
TOKENIZED_DIR    = '/content/output/tokenized/'
FORWARD_BATCH_SIZE = 100
NPROC = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZED_DIR, exist_ok=True)
print('Paths set')

In [ ]:
# Cell 4: Load and validate h5ad
adata = ad.read_h5ad(INPUT_FILE)
print(f'Shape          : {adata.shape}  (cells x genes)')
print(f'Sample cell IDs: {adata.obs_names[:3].tolist()}')
print(f'obs columns    : {adata.obs.columns.tolist()}')
print(f'var columns    : {adata.var.columns.tolist()}')
print(f'n_counts min   : {adata.obs["n_counts"].min():.1f}')

assert adata.var_names[0].startswith('ENSG'), 'var_names must be Ensembl IDs'
assert 'ensembl_id' in adata.var.columns, 'ensembl_id column missing from var'
assert (adata.obs['n_counts'] == 0).sum() == 0, 'Found zero-count cells'
print('Validation passed')

In [ ]:
# Cell 5: Tokenize
h5ad_copy = os.path.join(TOKENIZED_DIR, os.path.basename(INPUT_FILE))
if not os.path.exists(h5ad_copy):
    shutil.copy(INPUT_FILE, h5ad_copy)
    print('Copied h5ad to tokenizer directory')

tk = TranscriptomeTokenizer(
    custom_attr_name_dict={},
    nproc=NPROC,
    model_input_size=2048,
    special_token=True,
    gene_median_file=GENE_MEDIAN_FILE,
    token_dictionary_file=TOKEN_DICT_FILE,
)

tk.tokenize_data(
    data_directory=TOKENIZED_DIR,
    output_directory=TOKENIZED_DIR,
    output_prefix='melanoma_tcells',
    file_format='h5ad',
)

tokenized_path = os.path.join(TOKENIZED_DIR, 'melanoma_tcells.dataset')
print(f'Tokenized dataset: {tokenized_path}')

In [ ]:
# Cell 6: Extract embeddings
embex = EmbExtractor(
    model_type='Pretrained',
    num_classes=0,
    emb_mode='cell',
    cell_emb_style='mean_pool',
    filter_data=None,
    max_ncells=None,
    emb_layer=-1,
    emb_label=[],
    forward_batch_size=FORWARD_BATCH_SIZE,
    nproc=NPROC,
)

emb_df = embex.extract_embs(
    model_directory=GENEFORMER_DIR,
    input_data_file=tokenized_path,
    output_directory=OUTPUT_DIR,
    output_prefix='melanoma_tcells',
)
print(f'Embeddings shape: {emb_df.shape}')

In [ ]:
# Cell 7: Attach original cell IDs and save
original_cell_ids = adata.obs_names.tolist()

if len(original_cell_ids) != len(emb_df):
    raise ValueError(
        f'Cell count mismatch: h5ad has {len(original_cell_ids)} cells '
        f'but embeddings have {len(emb_df)} rows.'
    )

emb_df.index = original_cell_ids
print(f'Sample cell IDs: {emb_df.index[:3].tolist()}')

emb_adata = ad.AnnData(
    X   = emb_df.values.astype(np.float32),
    obs = pd.DataFrame(
        {'n_counts': adata.obs['n_counts'].values},
        index=original_cell_ids
    ),
    var = pd.DataFrame(
        index=[f'dim_{i}' for i in range(emb_df.shape[1])]
    ),
)

print(f'Output shape: {emb_adata.shape}  (cells x embedding dims)')
print(f'obs_names sample: {emb_adata.obs_names[:3].tolist()}')

emb_adata.write_h5ad(OUTPUT_H5AD, compression='gzip')
print(f'Saved: {OUTPUT_H5AD}')
print('Done! Download melanoma_tcells_embeddings.h5ad from the file panel on the left.')